In [ ]:
import os
from collections import OrderedDict

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import matplotlib.pyplot as plt
import numpy as np
import cv2

from PIL import Image


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ========================================
# IMAGE CONFIG
# ========================================
IMAGE_PATH = "../../results/Shampoo_NOBGR_pix2pix_StructCond_V1_Stage23_COMPLETESyn/test_latest/images_fake/deb1f5bb-2026-03-03_15-40-15-784_te_000095_fake_B.png"
#IMAGE_PATH = "../../results/Shampoo_NOBGR_pix2pix_StructCond_V1_Stage23_COMPLETESyn/test_latest/images_fake/9e487d1b-2026-03-03_16-54-56-681_te_000063_fake_B.png"
#IMAGE_PATH = "../../results/Shampoo_NOBGR_pix2pix_StructCond_V1_Stage23_COMPLETESyn/test_latest/images_fake/1de1b63b-2026-03-02_16-10-48-409_te_000099_fake_B.png"
#IMAGE_PATH = "../../results/Shampoo_NOBGR_pix2pix_StructCond_V1_Stage23_COMPLETESyn/test_latest/images_fake/4efcab4e-2026-03-02_13-49-07-525_te_000036_fake_B.png"




#IMAGE_PATH = "../../data/interim/Stage1/gray_clahe_1500x1000_noborder_aug/2026-01-21_10-38-24-999_aug01.png"
#IMAGE_PATH = "../../data/interim/Stage2/gray_clahe_1500x1000_noborder_aug/2026-01-21_15-46-35-205_aug01.png"
#IMAGE_PATH = "../../data/interim/Stage2/gray_clahe_1500x1000_noborder_aug/2026-01-21_15-59-19-201_aug01.png"
#IMAGE_PATH = "../../data/interim/Stage2/color_clahe_1500x1000_noborder_aug/2026-01-21_15-46-35-205_aug01.png"
#MAGE_PATH = "../../data/interim/Stage1/color_clahe_1500x1000_noborder_aug/2026-01-21_10-38-24-999_aug01.png"

#IMAGE_PATH = "../../data/raw/SHAMPOOBLADEWITHTRAY_COMPLETE/3d5018a3-2026-03-03_16-47-04-096.png"
#IMAGE_PATH = "../../data/raw/SHAMPOOBLADEWITHTRAY_COMPLETE/0c52233c-2026-03-03_16-03-48-194.png"
#IMAGE_PATH = "../../results/Shampoo_NOBGR_pix2pix_StructCond_V1_Stage23_COMPLETESyn/test_latest/images_real/5a5781af-2026-03-03_16-44-40-914_te_000061_real_B.png"



MODEL_PATH = "../../models/classifier/SHAMPOOBLADEINTRAY_COMPLETEV2/gray_multihead_stable/checkpoints/train_best.pt"

IMAGE_SIZE = 1024
GRAY_MEAN = (0.5,)
GRAY_STD = (0.25,)

SPATIAL_CLASSES = ["isolated", "overlap"]
THREAT_CLASSES = ["non_contraband", "contraband"]

# ========================================
# TRANSFORM
# ========================================
transform = T.Compose([
    T.Grayscale(num_output_channels=1),
    T.Resize(int(IMAGE_SIZE * 1.10)),
    T.CenterCrop(IMAGE_SIZE),
    T.ToTensor(),
    T.Normalize(GRAY_MEAN, GRAY_STD),
])

# ========================================
# MODEL
# ========================================
class SimpleCNN_MultiHead(nn.Module):
    def __init__(self, in_channels=1):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(256, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(), nn.MaxPool2d(2),
        )

        self.gap = nn.AdaptiveAvgPool2d((1, 1))

        self.shared_fc = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Dropout(0.35),
        )

        self.spatial_head = nn.Linear(512, 2)
        self.threat_head = nn.Linear(512, 2)

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        x = x.view(x.size(0), -1)
        x = self.shared_fc(x)

        spatial_logits = self.spatial_head(x)
        threat_logits = self.threat_head(x)

        return spatial_logits, threat_logits

# ========================================
# IMAGE HELPERS
# ========================================
def make_black_border_white(pil_img, threshold=15):
    img = pil_img.convert("L")
    arr = np.array(img)

    border_mask = arr <= threshold
    arr[border_mask] = 255

    return Image.fromarray(arr).convert("L")

def crop_white_border(pil_img, threshold=252, margin=0):
    img = pil_img.convert("L")
    arr = np.array(img)

    mask = arr < threshold

    if not mask.any():
        return pil_img

    ys, xs = np.where(mask)

    left = max(xs.min() - margin, 0)
    right = min(xs.max() + margin, pil_img.size[0] - 1)
    top = max(ys.min() - margin, 0)
    bottom = min(ys.max() + margin, pil_img.size[1] - 1)

    return pil_img.crop((left, top, right + 1, bottom + 1))

def prepare_image(image_path):
    img_raw = Image.open(image_path)
    print("Original image mode:", img_raw.mode)
    print("Original image size:", img_raw.size)

    img = make_black_border_white(img_raw, threshold=15)
    img = crop_white_border(img, threshold=252, margin=0)

    print("Processed display size:", img.size)

    plt.figure(figsize=(7, 5))
    plt.imshow(img, cmap="gray", vmin=0, vmax=255)
    plt.axis("off")
    plt.title("Generated image input")
    plt.show()

    return img

# ========================================
# LOAD MODEL SAFELY
# ========================================
model = SimpleCNN_MultiHead(in_channels=1).to(DEVICE)

try:
    state = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True)
except TypeError:
    state = torch.load(MODEL_PATH, map_location=DEVICE)

if isinstance(state, dict):
    if "model_state" in state:
        state = state["model_state"]
    elif "model_state_dict" in state:
        state = state["model_state_dict"]

if any(k.startswith("module.") for k in state.keys()):
    state = {k.replace("module.", "", 1): v for k, v in state.items()}

model.load_state_dict(state, strict=True)
model.eval()

print("Loaded model:", MODEL_PATH)

img = prepare_image(IMAGE_PATH)
x = transform(img).unsqueeze(0).to(DEVICE)

with torch.no_grad():
    spatial_logits, threat_logits = model(x)

    spatial_probs = torch.softmax(spatial_logits, dim=1).squeeze(0).cpu().numpy()
    threat_probs = torch.softmax(threat_logits, dim=1).squeeze(0).cpu().numpy()

    spatial_id = int(np.argmax(spatial_probs))
    threat_id = int(np.argmax(threat_probs))

print("Spatial:", SPATIAL_CLASSES[spatial_id], spatial_probs)
print("Threat:", THREAT_CLASSES[threat_id], threat_probs)

# ========================================
# PREDICT
# ========================================
@torch.no_grad()
def predict_generated_image(image_path):
    img = prepare_image(image_path)

    x = transform(img).unsqueeze(0).to(DEVICE)

    print("Input tensor shape:", x.shape)
    print("Input tensor min/max:", x.min().item(), x.max().item())

    spatial_logits, threat_logits = model(x)

    spatial_probs = torch.softmax(spatial_logits, dim=1).squeeze(0).cpu().numpy()
    threat_probs = torch.softmax(threat_logits, dim=1).squeeze(0).cpu().numpy()

    spatial_id = int(np.argmax(spatial_probs))
    threat_id = int(np.argmax(threat_probs))

    spatial_pred = SPATIAL_CLASSES[spatial_id]
    threat_pred = THREAT_CLASSES[threat_id]

    print("\n===== GENERATED IMAGE CNN EVALUATION =====")

    print("\nSpatial head:")
    print("  Class meaning: 0=isolated, 1=overlap")
    print(f"  Prediction: {spatial_pred} (class {spatial_id})")
    print(f"  isolated: {spatial_probs[0]:.4f}")
    print(f"  overlap : {spatial_probs[1]:.4f}")

    print("\nThreat head:")
    print("  Class meaning: 0=non_contraband, 1=contraband")
    print(f"  Prediction: {threat_pred} (class {threat_id})")
    print(f"  non_contraband: {threat_probs[0]:.4f}")
    print(f"  contraband    : {threat_probs[1]:.4f}")

    print("\nInterpretation:")
    if spatial_pred == "overlap":
        print("  Spatial: Objects appear overlapping / interacting.")
    else:
        print("  Spatial: Objects appear isolated / separated.")

    if threat_pred == "contraband":
        print("  Threat: Model detects contraband-like features.")
    else:
        print("  Threat: Model detects non-contraband-like features.")

    return {
        "image_path": image_path,
        "spatial_pred": spatial_pred,
        "spatial_probs": spatial_probs,
        "threat_pred": threat_pred,
        "threat_probs": threat_probs,
    }

result = predict_generated_image(IMAGE_PATH)

In [ ]:
# ========================================
# MULTI-HEAD GRAD-CAM FOR GENERATED IMAGE
# ========================================
import os
from collections import OrderedDict

import torch.nn.functional as F
import cv2
import numpy as np
import matplotlib.pyplot as plt

# ========================================
# CONFIG
# ========================================
GRADCAM_ALPHA = 0.40
SAVE_GRADCAM = False
GRADCAM_SAVE_DIR = "./gradcam_outputs_generated_multihead"

SHOW_ALL_LAYERS = True
SHOW_DETAILED_TRIPLETS = True

# Choose target:
# "predicted" = Grad-CAM for predicted class
# or manually use:
# SPATIAL_TARGET_CLASS = 1  # 0=isolated, 1=overlap
# THREAT_TARGET_CLASS = 1   # 0=non_contraband, 1=contraband
SPATIAL_TARGET_CLASS = "predicted"
THREAT_TARGET_CLASS = "predicted"

# ========================================
# SAFETY CHECK
# ========================================
required_vars = [
    "model", "x", "img",
    "SPATIAL_CLASSES", "THREAT_CLASSES",
    "spatial_probs", "threat_probs",
    "spatial_id", "threat_id",
]

missing = [v for v in required_vars if v not in globals()]

if missing:
    raise RuntimeError(
        f"Missing variables: {missing}. Run the generated-image prediction cell first."
    )

# ========================================
# IMAGE HELPERS
# ========================================
def denormalize_gray_tensor(x_tensor):
    """
    x_tensor: [1, 1, H, W]
    returns: H x W x 3 image in 0-1 range
    """
    arr = x_tensor.detach().cpu().squeeze(0).squeeze(0).numpy()
    arr = arr * GRAY_STD[0] + GRAY_MEAN[0]
    arr = np.clip(arr, 0, 1)

    arr_rgb = np.stack([arr, arr, arr], axis=-1)
    return arr_rgb

def make_heatmap_rgb(cam_01):
    heatmap_u8 = np.uint8(np.clip(cam_01, 0, 1) * 255)
    heatmap_bgr = cv2.applyColorMap(heatmap_u8, cv2.COLORMAP_JET)
    heatmap_rgb = cv2.cvtColor(heatmap_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    return heatmap_rgb

def overlay_heatmap(base_img_01, heatmap_rgb_01, alpha=0.40):
    overlay = (1 - alpha) * base_img_01 + alpha * heatmap_rgb_01
    return np.clip(overlay, 0, 1)

# ========================================
# GET CONV LAYERS
# ========================================
def get_all_conv_layers(model):
    conv_layers = OrderedDict()

    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            conv_layers[name] = module

    if len(conv_layers) == 0:
        raise RuntimeError("No Conv2d layers found in model")

    return conv_layers

conv_layers = get_all_conv_layers(model)

print("Conv layers found:")
for name in conv_layers.keys():
    print(" ", name)

# ========================================
# GRAD-CAM CORE FOR MULTI-HEAD MODEL
# ========================================
def compute_gradcam_for_layer_multihead(
    model,
    x_tensor,
    target_layer,
    head_name,
    target_class=None,
):
    """
    head_name: "spatial" or "threat"
    target_class:
      spatial: 0=isolated, 1=overlap
      threat : 0=non_contraband, 1=contraband
    """

    model.eval()

    activations = []
    gradients = []

    def forward_hook(module, inp, out):
        activations.append(out.detach())

    def backward_hook(module, grad_input, grad_output):
        gradients.append(grad_output[0].detach())

    h1 = target_layer.register_forward_hook(forward_hook)
    h2 = target_layer.register_full_backward_hook(backward_hook)

    try:
        spatial_logits, threat_logits = model(x_tensor)

        if head_name == "spatial":
            logits = spatial_logits
            class_names = SPATIAL_CLASSES
        elif head_name == "threat":
            logits = threat_logits
            class_names = THREAT_CLASSES
        else:
            raise ValueError("head_name must be 'spatial' or 'threat'")

        probs = torch.softmax(logits, dim=1)
        pred_id = int(probs.argmax(dim=1).item())

        if target_class is None:
            target_class = pred_id

        score = logits[:, int(target_class)].sum()

        model.zero_grad(set_to_none=True)
        score.backward()

        acts = activations[0]
        grads = gradients[0]

        weights = grads.mean(dim=(2, 3), keepdim=True)
        cam = (weights * acts).sum(dim=1, keepdim=True)
        cam = F.relu(cam)

        cam = cam.squeeze().detach().cpu().numpy().astype(np.float32)

        if cam.max() > 0:
            cam = cam / (cam.max() + 1e-8)

        H, W = x_tensor.shape[2], x_tensor.shape[3]
        cam_resized = cv2.resize(cam, (W, H), interpolation=cv2.INTER_LINEAR)

        return {
            "cam": cam_resized,
            "pred_id": pred_id,
            "pred_name": class_names[pred_id],
            "target_class": int(target_class),
            "target_name": class_names[int(target_class)],
            "probs": probs.squeeze(0).detach().cpu().numpy(),
        }

    finally:
        h1.remove()
        h2.remove()

def compute_gradcam_all_layers_multihead(model, x_tensor, head_name, target_class=None):
    conv_layers = get_all_conv_layers(model)
    results = []

    for layer_name, layer_module in conv_layers.items():
        out = compute_gradcam_for_layer_multihead(
            model=model,
            x_tensor=x_tensor,
            target_layer=layer_module,
            head_name=head_name,
            target_class=target_class,
        )

        out["layer_name"] = layer_name
        results.append(out)

    return results

# ========================================
# DISPLAY HELPERS
# ========================================
def show_gradcam_grid(head_name, base_img_01, gradcam_results):
    n = len(gradcam_results)
    cols = 3
    rows = int(np.ceil(n / cols))

    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = np.array(axes).reshape(-1)

    for ax, result in zip(axes, gradcam_results):
        layer_name = result["layer_name"]
        cam_01 = result["cam"]

        heatmap_rgb = make_heatmap_rgb(cam_01)
        overlay = overlay_heatmap(base_img_01, heatmap_rgb, alpha=GRADCAM_ALPHA)

        ax.imshow(overlay)
        ax.set_title(
            f"{head_name.upper()}\n"
            f"{layer_name}\n"
            f"Target: {result['target_name']}"
        )
        ax.axis("off")

    for ax in axes[len(gradcam_results):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

def show_gradcam_detailed_triplets(head_name, base_img_01, gradcam_results):
    for result in gradcam_results:
        layer_name = result["layer_name"]
        cam_01 = result["cam"]

        heatmap_rgb = make_heatmap_rgb(cam_01)
        overlay = overlay_heatmap(base_img_01, heatmap_rgb, alpha=GRADCAM_ALPHA)

        fig, axes = plt.subplots(1, 3, figsize=(12, 4))

        axes[0].imshow(base_img_01)
        axes[0].set_title("Input")
        axes[0].axis("off")

        axes[1].imshow(heatmap_rgb)
        axes[1].set_title(f"{head_name.upper()} — {layer_name}\nHeatmap")
        axes[1].axis("off")

        axes[2].imshow(overlay)
        axes[2].set_title(
            f"Overlay\n"
            f"Target: {result['target_name']}\n"
            f"Pred: {result['pred_name']}"
        )
        axes[2].axis("off")

        plt.tight_layout()
        plt.show()

def save_gradcam_triplets(head_name, base_img_01, gradcam_results, save_root):
    os.makedirs(save_root, exist_ok=True)

    for result in gradcam_results:
        layer_name = result["layer_name"].replace(".", "_")
        cam_01 = result["cam"]

        heatmap_rgb = make_heatmap_rgb(cam_01)
        overlay = overlay_heatmap(base_img_01, heatmap_rgb, alpha=GRADCAM_ALPHA)

        fig, axes = plt.subplots(1, 3, figsize=(12, 4))

        axes[0].imshow(base_img_01)
        axes[0].set_title("Input")
        axes[0].axis("off")

        axes[1].imshow(heatmap_rgb)
        axes[1].set_title(f"{layer_name} heatmap")
        axes[1].axis("off")

        axes[2].imshow(overlay)
        axes[2].set_title(
            f"{head_name.upper()}\n"
            f"Target: {result['target_name']}\n"
            f"Pred: {result['pred_name']}"
        )
        axes[2].axis("off")

        plt.tight_layout()

        out_path = os.path.join(
            save_root,
            f"{head_name}_{layer_name}_gradcam.png"
        )

        plt.savefig(out_path, dpi=200, bbox_inches="tight")
        plt.close(fig)

# ========================================
# RUN GRAD-CAM
# ========================================
base_img_01 = denormalize_gray_tensor(x)

if SPATIAL_TARGET_CLASS == "predicted":
    spatial_target = spatial_id
else:
    spatial_target = int(SPATIAL_TARGET_CLASS)

if THREAT_TARGET_CLASS == "predicted":
    threat_target = threat_id
else:
    threat_target = int(THREAT_TARGET_CLASS)

print("\n===== MULTI-HEAD GRAD-CAM =====")

print("\nSpatial target:")
print(f"  Target class: {spatial_target} = {SPATIAL_CLASSES[spatial_target]}")
print(f"  Predicted   : {spatial_id} = {SPATIAL_CLASSES[spatial_id]}")
print(f"  Prob isolated: {spatial_probs[0]:.4f}")
print(f"  Prob overlap : {spatial_probs[1]:.4f}")

spatial_gradcam_results = compute_gradcam_all_layers_multihead(
    model=model,
    x_tensor=x,
    head_name="spatial",
    target_class=spatial_target,
)

show_gradcam_grid(
    head_name="spatial",
    base_img_01=base_img_01,
    gradcam_results=spatial_gradcam_results,
)

if SHOW_DETAILED_TRIPLETS:
    show_gradcam_detailed_triplets(
        head_name="spatial",
        base_img_01=base_img_01,
        gradcam_results=spatial_gradcam_results,
    )

print("\nThreat target:")
print(f"  Target class: {threat_target} = {THREAT_CLASSES[threat_target]}")
print(f"  Predicted   : {threat_id} = {THREAT_CLASSES[threat_id]}")
print(f"  Prob non_contraband: {threat_probs[0]:.4f}")
print(f"  Prob contraband    : {threat_probs[1]:.4f}")

threat_gradcam_results = compute_gradcam_all_layers_multihead(
    model=model,
    x_tensor=x,
    head_name="threat",
    target_class=threat_target,
)

show_gradcam_grid(
    head_name="threat",
    base_img_01=base_img_01,
    gradcam_results=threat_gradcam_results,
)

if SHOW_DETAILED_TRIPLETS:
    show_gradcam_detailed_triplets(
        head_name="threat",
        base_img_01=base_img_01,
        gradcam_results=threat_gradcam_results,
    )

if SAVE_GRADCAM:
    save_gradcam_triplets(
        head_name="spatial",
        base_img_01=base_img_01,
        gradcam_results=spatial_gradcam_results,
        save_root=os.path.join(GRADCAM_SAVE_DIR, "spatial"),
    )

    save_gradcam_triplets(
        head_name="threat",
        base_img_01=base_img_01,
        gradcam_results=threat_gradcam_results,
        save_root=os.path.join(GRADCAM_SAVE_DIR, "threat"),
    )

    print(f"\nSaved Grad-CAM outputs to: {GRADCAM_SAVE_DIR}")

print("\n=======================================\n")